# 08 · Query embedding, search, retriever grid, rerankers and agentic runs

Plan sections 4 and 5, steps 4–7. Every method runs on both indexes, with the gate off and on (T4: gate on only), and the expansion methods at the 4k, 8k and 16k budgets: 110 runs for `params_version v1`.

| step | what | where | time |
|---|---|---|---|
| 4 | query embeddings (questions, HyDE passages) | CPU `encoder` | minutes |
| 5 | each signal's top-50 per index and query → `data/cache/search.jsonl` | Qdrant + client-side ColBERT | minutes |
| 6 | the retriever grid without rerankers and agent | CPU, from cached lists | minutes |
| 7a | X1 / X2 rerankers | CPU `encoder` | **hours** (~1 min per question for bge, more for jina) |
| 7b | T4 agent | vLLM + `encoder` | ~1–2 h |

Everything is cached and every run resumes from `partial.jsonl`, so the notebook can be interrupted and re-run. Runs whose `hits.parquet` already exists are skipped.

**Before running:** `docker compose up -d qdrant encoder`. vLLM (`serve.retrieval.yaml`) is needed only for LLM calls that notebook 07 did not cache, which in practice means T4.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import json, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from csrag import runs
from csrag.corpus import COLLECTIONS, Corpus
from csrag.gate import GateParams
from csrag.retrievers import Pipeline, Resources, check_rrf, grid, method_config, prompt_hashes
from csrag.retrievers.base import PARAMS
from csrag.retrievers.qdrant_search import Query

corpus = Corpus(DATA)
res = Resources.connect(corpus, DATA)
print("encoder:", {k: v["revision"][:10] for k, v in res.encoder.wait_ready()["models"].items()})
for index in corpus.indexes():
    n = res.client.get_collection(COLLECTIONS[index]).points_count
    assert n == len(corpus.units(index)), f"{index}: {n} points, run notebook 05"
    assert not res.colbert(index).missing, f"{index}: ColBERT vectors missing, run notebook 04"
print("Qdrant collections and ColBERT indexes ready")

In [ ]:
questions = [json.loads(l) for l in open(DATA / "eval" / "questions.jsonl", encoding="utf-8")]
texts = [q["question"] for q in questions]
labels = {q["question_id"]: q["engine_type"] for q in questions}  # passed to F0 (the oracle) only
prompts = prompt_hashes()
configs = grid("v1", prompts=prompts)
pd.Series([c["method"] for c in configs]).value_counts().sort_index().rename("runs").to_frame().T

## Step 4: query embeddings

Questions and HyDE passages, in dense, sparse and ColBERT form (`data/embeddings/queries_*.npz`, shared by both indexes). The HyDE passages come from the LLM cache filled in notebook 07. BM25 query vectors are computed on the fly (weight 1 per stem). Agent search queries are embedded when T4 runs.

In [ ]:
t0 = time.time()
q_emb = res.qemb["questions"].get(texts, res.encoder)
passages = [res.llm.hyde(t)[0] for t in texts]  # cached in notebook 07
h_emb = res.qemb["hyde"].get(passages, res.encoder)
print(len(q_emb), "question and", len(h_emb), "HyDE embeddings in", round(time.time() - t0, 1), "s")

## Step 5: search

Each signal's raw top-50 for every question and index, with scores and latency, into `data/cache/search.jsonl`. `dense_exact` repeats the dense search with `exact=True` (the ANN check). Filtered (F0, F1), HyDE (Q1) and keyword (Q2) searches fill the same cache during the grid.

In [ ]:
SIGNALS = ["bm25", "bge_sparse", "dense", "dense_exact", "colbert"]
for index in corpus.indexes():
    p = Pipeline(method_config("D1", index, False), res)
    for t in tqdm(texts, desc=index):
        q = p._query(t)
        for sig in SIGNALS:
            p.searcher.search(index, sig, q, None, 50)
print(len(res.caches.search), "cached search lists")

### Checks: ANN vs exact, and client-side RRF vs Qdrant's RRF (plan 8)

In [ ]:
rows = []
for index in corpus.indexes():
    p = Pipeline(method_config("H1", index, False), res)
    ann_same, rrf_ok = 0, 0
    for t in texts:
        q = p._query(t)
        a = p.searcher.search(index, "dense", q, None, 50)["hits"]
        b = p.searcher.search(index, "dense_exact", q, None, 50)["hits"]
        ann_same += [u for u, _ in a] == [u for u, _ in b]
        r = check_rrf(p, t)
        rrf_ok += r["same_set"] and r["same_scores"] and r["same_order_ignoring_ties"]
    rows.append({"index": index, "questions": len(texts), "dense HNSW == exact": ann_same, "RRF == Qdrant RRF (k=61)": rrf_ok})
pd.DataFrame(rows)

## Step 6: the retriever grid (fast loop)

Every configuration except the rerankers (X1, X2) and the agent (T4), with the gate off and on, from the cached lists. LLM outputs (Q1, Q2, F1) come from the cache.

In [ ]:
def execute(cfgs, desc):
    ids = []
    for cfg in tqdm(cfgs, desc=desc):
        p = Pipeline(cfg, res, labels=labels if cfg["method"] == "F0" else None)
        ids.append(runs.execute(cfg, p.run, questions, res.meta(cfg["index"])))
    return ids

fast = [c for c in configs if c["method"] not in ("X1", "X2", "T4")]
t0 = time.time()
fast_ids = execute(fast, "grid")
print(len(fast_ids), "runs in", round(time.time() - t0), "s")

### Gate checks (plan 8)

* gate-off lists equal the raw top-50 (here: the single-signal runs' top-20 equal the cached raw list's first 20);
* gate-on lists keep between `min_keep` and `max_keep` hits (or the whole list when it has fewer than 2 × `min_keep`), and are always a prefix of the gate-off list.

In [ ]:
ga = PARAMS["v1"]["gate_a"]
problems = []
for m, sig in {"L1": "bm25", "S1": "bge_sparse", "D1": "dense", "C1": "colbert"}.items():
    for index in corpus.indexes():
        off = runs.load_hits(runs.run_id_of(method_config(m, index, False, prompts=prompts)))
        on = runs.load_hits(runs.run_id_of(method_config(m, index, True, prompts=prompts)))
        st = {s["question_id"]: s for s in runs.load_stats(runs.run_id_of(method_config(m, index, True, prompts=prompts)))}
        p = Pipeline(method_config(m, index, False), res)
        for q in questions:
            qid = q["question_id"]
            a = list(off[off.question_id == qid].sort_values("rank")["unit_id"])
            b = list(on[on.question_id == qid].sort_values("rank")["unit_id"])
            raw = [u for u, _ in p.searcher.search(index, sig, p._query(q["question"]), None, 50)["hits"]]
            g = st[qid]["gates"][0]
            if a != raw[:20]:
                problems.append((m, index, qid, "gate-off != raw top-20"))
            if b != a[: len(b)]:
                problems.append((m, index, qid, "gate-on is not a prefix"))
            if g["n_in"] >= 2 * ga.min_keep and not (ga.min_keep <= g["n_kept"] <= ga.max_keep):
                problems.append((m, index, qid, f"kept {g['n_kept']}"))
print(len(problems), "problems", problems[:10])

## Step 7a: rerankers (CPU, slow)

Both rerankers get H1's fused list (at most 50, shorter when gate A is on) in the same order and return at most 20. First, the truncation check: the longest base unit plus the longest question goes through both models without truncation (bge `max_length` 8192, jina `max_doc_length` 8192).

In [ ]:
longest = max(corpus.units("units").values(), key=lambda u: corpus.n_tokens(u["text"]))
q_long = max(texts, key=len)
tb = res.encoder.tokens("bge", q_long, [longest["text"]])
tj = res.encoder.tokens("jina", q_long, [longest["text"]])
print(longest["unit_id"], "| bge pair tokens", tb["pairs"][0], "/", tb["max_length"], "| jina doc tokens", tj["documents"][0], "/", tj["max_doc_length"])
assert tb["pairs"][0] <= tb["max_length"] and tj["documents"][0] <= tj["max_doc_length"]

In [ ]:
rerank_cfgs = [c for c in configs if c["method"] in ("X1", "X2")]
t0 = time.time()
rerank_ids = execute(rerank_cfgs, "rerankers")
print(len(rerank_ids), "reranker runs in", round((time.time() - t0) / 60), "min")

## Step 7b: agentic runs (T4, vLLM + encoder)

Gate on only, at the three budgets, on both indexes. Each agent step is cached in `data/cache/llm.jsonl`; a re-run replays the cache.

In [ ]:
agent_cfgs = [c for c in configs if c["method"] == "T4"]
t0 = time.time()
agent_ids = execute(agent_cfgs, "agent")
print(len(agent_ids), "agent runs in", round((time.time() - t0) / 60), "min")
paths = pd.DataFrame([{"run_id": r, "question_id": s["question_id"], "path": " → ".join(s["agent"]["path"])}
                      for r in agent_ids for s in runs.load_stats(r)])
paths.groupby("run_id")["path"].value_counts().groupby(level=0).head(5).rename("questions").to_frame()

## Verification (plan 8)

* **Reproducibility**: re-running one config reproduces the same `hits.parquet` (temperature 0, cached LLM outputs).
* **No label leakage**: no retriever except F0 reads a label; the interface only receives `question_id` and `text`.
* **One hop**: no context in T3, T2+T3 or T4 holds hop-2 content, and every hop-1 item's source is hop-0 content of the same question.

In [ ]:
import tempfile
for cfg in [method_config("H1", "win500", True, prompts=prompts), method_config("T4", "units", True, budget=8192, prompts=prompts)]:
    rid = runs.run_id_of(cfg)
    with tempfile.TemporaryDirectory() as tmp:
        runs.execute(cfg, Pipeline(cfg, res).run, questions, res.meta(cfg["index"]), root=tmp)
        same = runs.load_hits(rid, tmp).equals(runs.load_hits(rid))
    print(rid, "reproduced:", same)

In [ ]:
import inspect
from csrag.retrievers import Pipeline as _P
assert list(inspect.signature(_P.run).parameters) == ["self", "question_id", "text"]
for m in ("H1", "F1", "T1", "X1"):
    assert Pipeline(method_config(m, "units", True, prompts=prompts), res, labels=labels).labels is None
label_fields = ("question_type", "engine_type", "gold_kind", "difficulty", "mentions_section_id", "gold", "answer")
src = "".join(inspect.getsource(__import__(f"csrag.retrievers.{m}", fromlist=["x"]))
              for m in ("agentic", "structural", "rerank", "fusion", "query_llm", "qdrant_search"))
print("label field names in retriever code:", [f for f in label_fields if f'["{f}"]' in src or f"['{f}']" in src])
print("only F0 reads labels: OK")

In [ ]:
bad = []
for rid in runs.list_runs(params_version="v1"):
    cfg = runs.load_config(rid)["config"]
    ex = cfg.get("expansion") or {}
    if ex.get("rule") not in ("refs", "parent+refs", "agent"):
        continue
    h = runs.load_hits(rid)
    h = h[h["in_context"]]
    if (~h["hop"].isin([0, 1])).any():
        bad.append((rid, "hop > 1"))
    for qid, g in h.groupby("question_id"):
        hop0 = set(g.loc[g["hop"] == 0, "unit_id"])
        src = set(g.loc[g["hop"] == 1, "source"])
        if not src <= hop0:
            bad.append((rid, qid, src - hop0))
print(len(bad), "one-hop violations", bad[:5])

In [ ]:
v1 = runs.list_runs(params_version="v1")
print(len(v1), "v1 runs (plan: ~110)")
pd.Series([runs.load_config(r)["retriever"] for r in v1]).value_counts().sort_index().rename("runs").to_frame().T

## v2: the one revision (plan 4.8)

After reading **all** v1 metrics in notebook 09, revise the parameters **once**, by hand, in `src/csrag/retrievers/base.py`:

```python
PARAMS["v2"] = revise("v1", gate_a=GateParams(tau=..., delta=..., min_keep=..., max_keep=...), ...)
```

Then run the whole grid again with `params_version="v2"`. Both versions are kept: v1 is the a-priori result, v2 the revised one (chosen after seeing v1 on the same questions).

In [ ]:
RUN_V2 = False
if RUN_V2:
    configs_v2 = grid("v2", prompts=prompts)
    v2_ids = execute([c for c in configs_v2 if c["method"] not in ("X1", "X2", "T4")], "grid v2")
    v2_ids += execute([c for c in configs_v2 if c["method"] in ("X1", "X2")], "rerankers v2")
    v2_ids += execute([c for c in configs_v2 if c["method"] == "T4"], "agent v2")